# XiTuner — GPU kill-risk gate (Colab)

**Purpose:** answer one question, fast — does LoRA fine-tuning produce a
behavior change obvious to the naked eye?

This exists because the same gate on CPU is not practical. Measured on a
6-core Ryzen 5 5500:

| base model | raw params | projected CPU run |
|---|---|---|
| SmolLM2-135M | 135M | 33 min |
| `gemma-3-270m` | 268M | 65 min |
| `gemma-4-E2B-it` | **5.12B** | **20.7 h** — and needs 20.5GB RAM vs 15.9GB available |

`E2B` means *Effective* 2B. The raw weight count is 5.12B, and training
touches raw weights, so on CPU the model never even loads.

**Two things to be clear about before starting:**

1. **Colab is not Google Cloud.** It does not satisfy the hackathon's
   "at least one Google Cloud infrastructure service" requirement. This
   notebook is a bridge for development while GCP credits are pending; the
   submission still needs training on Vertex AI with the orchestrator on
   Cloud Run.
2. **Colab sessions are ephemeral.** Everything on local disk disappears when
   the runtime is recycled. The last cell saves the adapter to Drive — do not
   skip it.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

## 1. Confirm a GPU is actually attached

Do not skip this. Colab silently gives a CPU runtime if no GPU is available,
and every timing number below would then be meaningless.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

import subprocess, sys
if subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
    sys.exit("No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.")
print("GPU present")

## 2. Install dependencies

`bitsandbytes` is what makes 4-bit (QLoRA) possible, which is what makes a
5.12B model fit in 16GB of VRAM.

In [ ]:
%pip install -q -U transformers peft trl datasets accelerate bitsandbytes pydantic python-dotenv

import transformers, peft, trl, datasets
print("transformers", transformers.__version__)
print("peft        ", peft.__version__)
print("trl         ", trl.__version__)
print("datasets    ", datasets.__version__)

## 3. Get the XiTuner code

Two options. Use **A** once the repo is on GitHub (which the submission needs
anyway). Use **B** today, before it is pushed.

No code is duplicated into this notebook on purpose: the notebook runs the
same modules as local development, so there is no second implementation to
keep in sync.

In [ ]:
# --- OPTION A: clone from GitHub (preferred once pushed) ---
REPO_URL = ""  # e.g. "https://github.com/<you>/xituner.git"

import os

if REPO_URL:
    !git clone -q $REPO_URL xituner
    os.chdir("xituner")
    print("cloned:", os.getcwd())
else:
    print("REPO_URL empty — use Option B below.")

In [ ]:
# --- OPTION B: upload a zip of the repo ---
# Build it locally from the project root:
#   python -m scripts.make_colab_zip
#
# Do NOT use PowerShell's Compress-Archive: it writes zip entries with
# backslashes, so Linux unzip creates a file literally named
# 'training\\config.py' instead of a training/ directory, and the import then
# fails with a ModuleNotFoundError that looks unrelated to the cause.
#
# Re-run the builder after any code change -- this uploads a snapshot.

import os

if not os.path.isdir("training"):
    from google.colab import files

    uploaded = files.upload()
    for name in uploaded:
        if name.endswith(".zip"):
            !unzip -q -o $name
    # The package dirs need __init__.py; the zip may omit empty-ish files.
    for pkg in ("training", "scripts"):
        os.makedirs(pkg, exist_ok=True)
        init = os.path.join(pkg, "__init__.py")
        if not os.path.exists(init):
            open(init, "w").close()

print("contents:", sorted(os.listdir(".")))
assert os.path.isdir("training"), "training/ not found — upload did not land"

## 4. Pick the base model

`gemma-4-E2B-it` is **ungated** — no Hugging Face token, no license click, for
you or for a judge reproducing this. `gemma-3-270m` is gated and needs a token,
so it is the friction-heavier option despite being smaller.

Do not demo a non-Gemma model: that forfeits the +0.2 Google-model bonus.

In [ ]:
import os

os.environ["BASE_MODEL"] = "google/gemma-4-E2B-it"
os.environ.pop("XITUNER_FORCE_CPU", None)  # we want the GPU here

# Only needed for gated repos (gemma-3-*). Leave empty for Gemma 4.
# os.environ["HF_TOKEN"] = "hf_..."

!python -m scripts.probe_model --models $BASE_MODEL

## 5. Build the scaffolding corpus

180 rows teaching a deliberately distinctive output shape:

```
Singkat: <one line>
Langkah:
1. ...
Catatan: kalau ragu, tanya penyuluh setempat.
```

Shape, not domain knowledge — a small model will not absorb agronomy from 180
examples, but it will absorb a template, and a template is what makes the
before/after unmistakable on video.

This corpus is synthetic scaffolding. Real messy data (WhatsApp exports, PDF
guides, field notes) replaces it once the gate passes, and anything synthetic
that survives into the submission must be disclosed.

In [ ]:
!python -m scripts.make_seed_corpus

## 6. Measure before trusting a plan

The original project plan claimed "one run finishes in minutes on CPU". That
was an assumption, and measurement falsified it. Measure here too — the
4-minute demo script depends on this number.

In [ ]:
!python -m scripts.benchmark_cpu --skip-timing

## 7. Train

`--load-in-4bit` is required: 5.12B raw parameters is ~20.5GB at float32 and
~10.2GB at bf16, versus 16GB of T4 VRAM that also has to hold activations and
gradients. 4-bit nf4 brings the weights to roughly 2.6GB.

Note what is *not* here: no LLM picks the hyperparameters, and no LLM decides
when to stop. Those come from a heuristic table and `EarlyStoppingCallback`.
Gemini's judgment is spent on corpus surgery, behavioral refereeing, and
diagnosis — work with no deterministic equivalent.

In [ ]:
!python -m training.train_lora --load-in-4bit --output-dir outputs/gate_gpu

## 8. The gate: base vs tuned, same prompts

Scored deterministically on structural signature match — no LLM judgment. The
Gemini Referee comes later and answers a harder question (*is the output
actually good*). This one only answers: **did anything change at all.**

Pass condition: tuned ≥75% signature, delta ≥50 points, and no degeneration.

This output is also the video's money shot. A loss curve is the worst visual
in existence; two columns where the left is nonsense and the right is correct
reads in ten seconds.

In [ ]:
!python -m scripts.compare_behavior --load-in-4bit --adapter-dir outputs/gate_gpu

## 9. Save the adapter before the runtime dies

Colab recycles runtimes and takes local disk with it. A LoRA adapter is small
(tens of MB), so there is no reason to lose one.

In [ ]:
import shutil, datetime, os

from google.colab import drive

drive.mount("/content/drive")

stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M")
dest = f"/content/drive/MyDrive/xituner/gate_gpu-{stamp}"
shutil.copytree("outputs/gate_gpu", dest, dirs_exist_ok=True)
print("saved ->", dest)
print(sorted(os.listdir(dest)))

## What comes next

**If the gate passed:** the premise holds. Move on to the agent layer —
Spec Compiler, Corpus Surgeon, Behavioral Referee, Diagnostician — and swap
the scaffolding corpus for real messy data.

**If it failed:** escalate in this order before writing any agent code.
1. more epochs / higher LoRA rank
2. larger seed corpus
3. larger base model (`gemma-4-E4B-it`, 8B raw)

Either way, this notebook is development scaffolding. The submission has to
show training on **Vertex AI** driven by an orchestrator on **Cloud Run** —
Colab does not satisfy the Google Cloud infrastructure requirement.